# 03 · Modelado y seguimiento (Fase 2)

Este notebook **solo importa los módulos de `src/`** (la lógica vive en el paquete, no aquí). Sirve para ejecutar el flujo, revisar resultados y consultar MLflow.

Prerrequisito: CSV de MiBici en `data/raw/<año>/datos_abiertos_*.csv` y `pip install -r requirements.txt`.

In [ ]:
import pandas as pd
import mlflow
from mlflow import MlflowClient

from src.config import load_params, resolve_path
from src.data.make_dataset import build_interim_dataset
from src.features.build_features import build_features_dataset
from src.models.train_model import run_training
from src.models.predict_model import load_model, predict

params = load_params()          # todo (rutas, semilla, hiperparámetros) sale de params.yaml
params['training']

## 1. Datos y features (módulos `make_dataset` y `build_features`)
Si ya existen los parquet se pueden omitir estas dos celdas.

In [ ]:
interim_file = build_interim_dataset(params)
features_file = build_features_dataset(params)
features = pd.read_parquet(features_file)
features.shape

In [ ]:
features.head()

## 2. Entrenamiento, ajuste y registro (`train_model.run_training`)
Ejecuta baselines, regresión lineal y XGBoost (RandomizedSearchCV con `TimeSeriesSplit`). Todo queda en MLflow y los modelos en el Model Registry.

In [ ]:
summary = run_training(params, features_file=features_file)
summary['champion']

In [ ]:
results = pd.DataFrame(summary['results'])
results.round(3)

## 3. Gráficas generadas (`reports/figures`)

In [ ]:
from IPython.display import Image, display
for name in ['model_comparison', 'mae_por_mes', 'mae_por_volumen', 'pred_vs_real']:
    display(Image(filename=str(resolve_path(params['paths']['figures']) / f'{name}.png')))

## 4. Consulta de MLflow
Para las gráficas comparativas interactivas: `make mlflow-ui` y abrir el experimento `mibici-demand-forecast` (seleccionar los runs `xgb-candidate-*` → *Chart view*).

In [ ]:
mlflow.set_tracking_uri(params['mlflow']['tracking_uri'])
exp = mlflow.get_experiment_by_name(params['mlflow']['experiment'])
runs = mlflow.search_runs([exp.experiment_id], filter_string="tags.candidate = 'true'")
runs[['tags.mlflow.runName', 'params.n_estimators', 'params.max_depth', 'params.learning_rate', 'params.objective', 'metrics.cv_mae']].sort_values('metrics.cv_mae').head(10)

In [ ]:
client = MlflowClient()
for rm in client.search_registered_models():
    print(rm.name, '| alias:', rm.aliases)
    for v in client.search_model_versions(f"name='{rm.name}'"):
        print('   versión', v.version, '| run', v.run_id[:8])

## 5. Inferencia con el modelo champion (`predict_model`)

In [ ]:
model = load_model(params)     # models/champion.joblib; también: load_model(params, 'models:/mibici-demand-xgboost@champion')
predict(model, features.tail(24 * 5), params).head()